# Product Recommendation Agent System (Tavily Search)

### What are we building?
A system with **4 AI agents** that work together to recommend the best product for you.

### Why 4 agents instead of 1?
Each agent is an **expert** at one thing. Just like in a company:
- Agent 0 = **Scout** → Finds products on the internet
- Agent 1 = **Engineer** → Researches technical specs
- Agent 2 = **Critic** → Reads and analyzes reviews
- Agent 3 = **Advisor** → Scores everything, gives final recommendation

### How does it flow?
```
Your Query → Agent 0 (find products) → Agent 1 (get specs) → Agent 2 (get reviews) → Agent 3 (rank & recommend)
```

### Tech Stack
- **Pydantic** — Forces LLM to return structured data (not random text)
- **LangChain** — Connects prompts to LLMs easily
- **Tavily** — AI-optimized search API (needs API key)
- **OpenAI GPT** — The brain (LLM)

**Constraint**: No orchestration frameworks (LangGraph, CrewAI, AutoGen) — pure Python.

---
# PART 1: Setup & Configuration

### What happens here?
1. Install required packages
2. Import all libraries
3. Load API keys from `.env` file (keeps secrets safe)
4. Create the LLM (AI brain) and search tool

In [1]:
# Uncomment and run ONCE to install packages
!pip install langchain langchain-openai langchain-community pydantic python-dotenv tavily-python

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\BhargavDharan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os                                    # Read environment variables
from datetime import date                    # Get today's date for queries
from typing import List, Optional            # Type hints for Pydantic

from dotenv import load_dotenv               # Load .env file
from pydantic import BaseModel, Field        # Structured output schemas

from langchain_openai import ChatOpenAI       # OpenAI LLM wrapper
from langchain_core.prompts import ChatPromptTemplate  # Prompt templates

from tavily import TavilyClient               # Tavily search API

C:\Users\BhargavDharan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# load_dotenv() reads .env file → makes keys available via os.getenv()
load_dotenv(override=True)

# Tavily = AI-optimized search engine. Gives cleaner results than Google.
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

# ChatOpenAI = OpenAI's GPT via LangChain
# temperature: 0=deterministic, 1=creative. 0.3 = focused but slightly creative
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

print("Setup complete!")

Setup complete!


### Experiment: Try changing the model
```python
llm = ChatOpenAI(model="gpt-4o", temperature=0.7)    # More creative, smarter model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # Very deterministic
```

In [4]:
llm = ChatOpenAI(model="gpt-5", temperature=0.5)

In [5]:
llm.invoke("Hello, world!")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 10, 'total_tokens': 28, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DlY4hsO2alHuC0KN8f6b9stXArurS', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e7daa-28f5-7c10-81de-f068aa5c1bbd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 18, 'total_tokens': 28, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [13]:
response = llm.invoke("What is LLM?")
print(response)

content='LLM can mean two common things:\n\n- Large Language Model: A type of AI trained on vast amounts of text to predict and generate words (tokens). Built mostly with transformer neural networks, they can answer questions, write text, translate, summarize, and generate code. Examples include GPT-4, Claude, and Llama. Strengths: versatile language skills; limits: can make factual errors (“hallucinations”) and lack true understanding.\n\n- LL.M. (Master of Laws): A postgraduate law degree, typically 1 year full-time. Often pursued after a first law degree to specialize (e.g., tax, IP, international law) or gain international credentials. Not usually required to practice law but can enhance expertise and career options.\n\nWhich one did you mean?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 420, 'prompt_tokens': 11, 'total_tokens': 431, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 

In [14]:
print(response.content)

LLM can mean two common things:

- Large Language Model: A type of AI trained on vast amounts of text to predict and generate words (tokens). Built mostly with transformer neural networks, they can answer questions, write text, translate, summarize, and generate code. Examples include GPT-4, Claude, and Llama. Strengths: versatile language skills; limits: can make factual errors (“hallucinations”) and lack true understanding.

- LL.M. (Master of Laws): A postgraduate law degree, typically 1 year full-time. Often pursued after a first law degree to specialize (e.g., tax, IP, international law) or gain international credentials. Not usually required to practice law but can enhance expertise and career options.

Which one did you mean?


In [15]:
print("\n" + "="*80)
print("LLM RESPONSE")
print("="*80)
print(response.content)
print("="*80)


LLM RESPONSE
LLM can mean two common things:

- Large Language Model: A type of AI trained on vast amounts of text to predict and generate words (tokens). Built mostly with transformer neural networks, they can answer questions, write text, translate, summarize, and generate code. Examples include GPT-4, Claude, and Llama. Strengths: versatile language skills; limits: can make factual errors (“hallucinations”) and lack true understanding.

- LL.M. (Master of Laws): A postgraduate law degree, typically 1 year full-time. Often pursued after a first law degree to specialize (e.g., tax, IP, international law) or gain international credentials. Not usually required to practice law but can enhance expertise and career options.

Which one did you mean?


In [11]:
!pip install rich

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\BhargavDharan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\BhargavDharan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [16]:
from rich.console import Console
from rich.panel import Panel

console = Console()

response = llm.invoke("Explain LLM in AI ?")

console.print(
    Panel(
        response.content,
        title="AI Response",
        expand=False
    )
)

╭────────────────────────────────────────────────── AI Response ──────────────────────────────────────────────────╮
│ An LLM (Large Language Model) is an AI system trained to understand and generate human-like text. At its core,  │
│ it’s a very large neural network that predicts the next word (more precisely, the next token) in a sequence,    │
│ having learned patterns from vast amounts of text.                                                              │
│                                                                                                                 │
│ Key ideas:                                                                                                      │
│ - What it does: Given a prompt, it continues the text in a way that fits the patterns it has learned. This      │
│ enables tasks like answering questions, summarizing, translating, drafting content, writing code, and more.     │
│ - How it works (simplified):                                                                                    │
│   - Tokenization: Text is split into tokens (words or subwords).                                                │
│   - Transformer architecture: Uses self-attention to weigh which parts of the input are most relevant when      │
│ generating each next token.                                                                                     │
│   - Pretraining: The model learns general language patterns by predicting the next token across huge text       │
│ corpora.                                                                                                        │
│   - Alignment/tuning: Additional steps (instruction tuning, reinforcement learning from human feedback) teach   │
│ it to follow directions, be helpful, and avoid unsafe outputs.                                                  │
│   - Inference: At use time, you provide a prompt within a context window; the model generates tokens, often     │
│ sampled with controls like temperature (creativity) and top-p (focus).                                          │
│ - Capabilities:                                                                                                 │
│   - Language tasks: Q&A, summarization, classification, information extraction, translation.                    │
│   - Reasoning and planning to a degree (chain-of-thought, tool use when connected to external systems).         │
│   - Coding assistance: completing functions, explaining code, generating tests.                                 │
│ - Strengths:                                                                                                    │
│   - Flexible across many domains without task-specific training.                                                │
│   - Effective with few examples (few-shot) or just clear instructions (zero-shot).                              │
│ - Limitations:                                                                                                  │
│   - Hallucinations: Can produce confident but incorrect facts.                                                  │
│   - Not a database: Lacks guaranteed up-to-date or verifiable knowledge unless connected to tools/search.       │
│   - Sensitivity to prompts and context length limits.                                                           │
│   - Bias and safety concerns reflecting training data.                                                          │
│   - Non-determinism: Same prompt can yield different outputs.                                                   │
│ - Using LLMs well:                                                                                              │
│   - Be specific about the task, audience, and constraints.                                                      │
│   - Provide examples or structure (few-shot prompting).                                                         │
│   - Ask for step-by-step reasoning or citations when n

---
# PART 2: Understanding Pydantic (Structured Output)

### The Problem
Without Pydantic, LLM returns unpredictable formats every time:
```
"ASUS ROG Strix is great..."           ← plain text
{"laptop": "ASUS"}                     ← random JSON
["ASUS", "Lenovo"]                     ← just a list
```

### The Solution
Pydantic defines a **strict schema**. The LLM is forced to follow it:
```python
class Product(BaseModel):
    brand: str    # MUST return a string
    model: str    # MUST return a string
```
Now LLM always returns: `{"brand": "ASUS", "model": "ROG Strix G16"}`

`Field(description=...)` tells the LLM **what to put** in each field.

In [17]:
# ======== AGENT 0: Discovery Agent Schema ========
# What Agent 0 returns: a list of products found on the web

class DiscoveredProduct(BaseModel):
    brand: str = Field(description="Product manufacturer (e.g., ASUS, Lenovo)")
    model: str = Field(description="Product model name (e.g., ROG Strix G16)")
    specs_hint: str = Field(description="Brief specs from search (e.g., RTX 4060, 16GB)")
    source: str = Field(description="URL where this product was found")
    reason: str = Field(description="Why this product matches the user query")

class DiscoveryOutput(BaseModel):
    category: str = Field(description="Detected category (e.g., Laptops)")
    budget: str = Field(description="Detected budget (e.g., Under 1,50,000)")
    use_case: str = Field(description="Detected use case (e.g., AI Development)")
    products: List[DiscoveredProduct] = Field(description="List of products (up to 5)")

print("Discovery schema defined. Fields:", list(DiscoveryOutput.model_fields.keys()))

Discovery schema defined. Fields: ['category', 'budget', 'use_case', 'products']


In [18]:
# ======== AGENT 1: Specification Agent Schema ========
# What Agent 1 returns: detailed specs for each product

class TechnicalSpecs(BaseModel):
    product_name: str = Field(description="Full product name (Brand + Model)")
    cpu: str = Field(description="Processor model and speed")
    gpu: str = Field(description="Graphics card model and VRAM")
    ram: str = Field(description="RAM capacity and type (e.g., 16GB DDR5)")
    storage: str = Field(description="Storage capacity and type (e.g., 1TB SSD)")
    display: str = Field(description="Display size, resolution, panel type")
    battery: str = Field(description="Battery capacity and estimated life")
    weight: str = Field(description="Device weight in kg")
    price: str = Field(description="Approximate price in INR")
    summary: str = Field(description="One-line AI/ML focused summary")

class SpecificationOutput(BaseModel):
    specs: List[TechnicalSpecs] = Field(description="Specs for all products")

print("Specification schema defined. Fields:", list(TechnicalSpecs.model_fields.keys()))

Specification schema defined. Fields: ['product_name', 'cpu', 'gpu', 'ram', 'storage', 'display', 'battery', 'weight', 'price', 'summary']


In [19]:
# ======== AGENT 2: Review Analysis Agent Schema ========
# What Agent 2 returns: sentiment, pros, cons for each product

class ProductReview(BaseModel):
    product_name: str = Field(description="Full product name")
    overall_sentiment: str = Field(description="Positive, Mixed, or Negative")
    average_rating: str = Field(description="Estimated rating out of 5")
    pros: List[str] = Field(description="Top 3-5 praised features")
    cons: List[str] = Field(description="Top 3-5 common complaints")
    common_issues: List[str] = Field(description="Recurring problems")
    expert_opinion: str = Field(description="Expert reviewer summary")

class ReviewOutput(BaseModel):
    reviews: List[ProductReview] = Field(description="Review analysis for all products")

print("Review schema defined.")

Review schema defined.


In [20]:
# ======== AGENT 3: Recommendation Agent Schema ========
# What Agent 3 returns: scores, rankings, final verdict

class ProductScore(BaseModel):
    product_name: str = Field(description="Full product name")
    performance_score: float = Field(description="Hardware capability (0-10)")
    value_score: float = Field(description="Value for money (0-10)")
    review_score: float = Field(description="User satisfaction (0-10)")
    ai_readiness_score: float = Field(description="AI/ML suitability (0-10)")
    overall_score: float = Field(description="Weighted overall (0-10)")
    rank: int = Field(description="Final rank (1 = best)")

class RecommendationOutput(BaseModel):
    top_pick: str = Field(description="#1 recommended product")
    top_pick_reason: str = Field(description="Why it's the top pick")
    scores: List[ProductScore] = Field(description="Scores for all products")
    trade_offs: str = Field(description="Trade-offs between top products")
    final_verdict: str = Field(description="Final advice (2-3 sentences)")
    confidence: str = Field(description="High, Medium, or Low")

print("All 4 agent schemas ready!")

All 4 agent schemas ready!


### Experiment: Test Pydantic yourself

In [21]:
# EXPERIMENT: Create a Pydantic object manually
test = DiscoveredProduct(
    brand="ASUS", model="ROG Strix G16", specs_hint="RTX 4060",
    source="https://example.com", reason="Great for AI"
)
print("Object:", test)
print("Dict:", test.model_dump())
print("Brand:", test.brand)

# TRY: What happens with missing fields? Uncomment below:
# bad = DiscoveredProduct(brand="ASUS")  # ValidationError!

Object: brand='ASUS' model='ROG Strix G16' specs_hint='RTX 4060' source='https://example.com' reason='Great for AI'
Dict: {'brand': 'ASUS', 'model': 'ROG Strix G16', 'specs_hint': 'RTX 4060', 'source': 'https://example.com', 'reason': 'Great for AI'}
Brand: ASUS


---
# PART 3: Understanding Search Tools

### What does the search tool do?
Searches the internet and returns results — like Googling in code.

### Why Tavily over regular Google?
- Google returns messy HTML → hard to parse
- Tavily returns **clean structured results** optimized for AI
- Returns: title, URL, content snippet, relevance score

In [ ]:
# 3 search functions — each for a different purpose

def search_products(query: str, max_results: int = 5) -> str:
    """General search. Returns formatted results as string for LLM."""
    try:
        results = tavily_client.search(query=query, max_results=max_results)
        content = ""
        for item in results["results"]:
            content += f"\nTitle: {item['title']}\nURL: {item['url']}\nContent: {item['content']}\n"
        return content
    except Exception as e:
        print(f"Search error: {e}")
        return ""

def search_specs(product_name: str) -> str:
    """Search for technical specs of a product."""
    return search_products(f"{product_name} full technical specifications price India {date.today().year}", 3)

def search_reviews(product_name: str) -> str:
    """Search for reviews and user opinions."""
    return search_products(f"{product_name} review pros cons user experience {date.today().year}", 3)

print("Search tools ready.")

### Experiment: Test search yourself

In [ ]:
# EXPERIMENT: Change the query and see what comes back
test_results = search_products("best laptops for AI development 2026 India", max_results=3)
print(test_results)

---
# PART 4: Agent 0 — Discovery Agent

### What does this agent do?
1. Takes your query → 2. Searches the web → 3. LLM extracts product names → 4. Returns structured list

### How is an agent built? (3 pieces)
```python
prompt = ChatPromptTemplate(...)                              # Instructions
chain  = prompt | llm.with_structured_output(Schema)         # Connect prompt → LLM
result = chain.invoke({"var1": "...", "var2": "..."})        # Run it
```

`chain.invoke()` fills variables into prompt → sends to LLM → returns Pydantic object

In [ ]:
# BUILD: Prompt template with {user_query} and {search_context} placeholders
discovery_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a product discovery expert.
Analyze the user query and search results. Extract up to 5 real product models.
Ignore generic articles — only include specific products with brand and model.
"""),
    ("human", "User Query:\n{user_query}\n\nSearch Results:\n{search_context}")
])

# BUILD: Chain = prompt → structured LLM
discovery_chain = discovery_prompt | llm.with_structured_output(DiscoveryOutput)
print("Discovery Agent built! Input vars:", discovery_prompt.input_variables)

In [ ]:
# RUN: Discovery Agent
user_query = "Best laptops for AI development under 150000 INR"
print(f"Query: {user_query}\n")

# Step 1: Search web
search_context = search_products(f"{user_query} top models India {date.today().year}", 10)

# Step 2: LLM extracts products from search results
discovery_result = discovery_chain.invoke({"user_query": user_query, "search_context": search_context})

# Step 3: Display
print(f"Category: {discovery_result.category} | Budget: {discovery_result.budget} | Use: {discovery_result.use_case}")
print(f"\nFound {len(discovery_result.products)} products:")
for i, p in enumerate(discovery_result.products, 1):
    print(f"  {i}. {p.brand} {p.model} — {p.specs_hint}")

In [ ]:
# EXPERIMENT: Inspect the structured output
print("Type:", type(discovery_result))
print("First product:", discovery_result.products[0].brand, discovery_result.products[0].model)
print("As dict:", discovery_result.model_dump())

---
# PART 5: Agent 1 — Specification Agent

### What does this agent do?
Takes product list from Agent 0 → searches specs for each → LLM extracts CPU/GPU/RAM/etc.

### Key Pattern: Data flows between agents
```
Agent 0 output (discovery_result.products) → becomes input for Agent 1
```

In [ ]:
# BUILD: Spec Agent
spec_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a hardware spec expert. Extract: CPU, GPU, RAM, Storage, Display, Battery, Weight, Price.
If not found, write "Not available". Add a one-line AI/ML summary.
"""),
    ("human", "Products:\n{product_list}\n\nResearch Data:\n{research_data}")
])
spec_chain = spec_prompt | llm.with_structured_output(SpecificationOutput)
print("Specification Agent built!")

In [ ]:
# RUN: Spec Agent
print("[Agent 1] Researching specs...\n")

# Prepare product list string from Agent 0 output
product_list_str = "\n".join(f"- {p.brand} {p.model}" for p in discovery_result.products)

# Search specs for EACH product
all_research = ""
for p in discovery_result.products:
    name = f"{p.brand} {p.model}"
    print(f"  Searching: {name}")
    all_research += f"\n--- {name} ---\n{search_specs(name)}\n"

# LLM extracts structured specs
spec_result = spec_chain.invoke({"product_list": product_list_str, "research_data": all_research})

print("\n" + "=" * 70 + "\nSPECIFICATION COMPARISON\n" + "=" * 70)
for s in spec_result.specs:
    print(f"\n{s.product_name}")
    print(f"  CPU: {s.cpu} | GPU: {s.gpu} | RAM: {s.ram}")
    print(f"  Storage: {s.storage} | Display: {s.display}")
    print(f"  Battery: {s.battery} | Weight: {s.weight} | Price: {s.price}")
    print(f"  Summary: {s.summary}")

---
# PART 5.5: Understanding Sentiment Analysis

Before we build the Review Agent, let's understand **what sentiment analysis is** and **how it works**.

---

## What is Sentiment Analysis?

Sentiment analysis is the process of **determining whether a piece of text expresses a positive, negative, or neutral opinion**.

Think of it like reading a product review and asking: *Is this person happy or unhappy with the product?*

### Real-world examples:

```
"This laptop is amazing! Great performance and battery life."  → POSITIVE
"Terrible build quality. Overheats within 30 minutes."         → NEGATIVE
"It's okay for the price. Nothing special."                    → NEUTRAL / MIXED
```

---

## Where is Sentiment Analysis Used?

| Use Case | Example |
|----------|---------|
| Product Reviews | Amazon ratings, Flipkart reviews |
| Social Media | Twitter/X brand monitoring |
| Customer Support | Auto-detect angry customers |
| Stock Market | News sentiment → predict stock movement |
| Our Project | Analyze laptop reviews to find pros/cons |

---

## How Does Sentiment Analysis Work?

There are **3 main approaches**:

### Approach 1: Rule-Based (Old Way)
- Maintain a list of positive words (`great`, `excellent`, `love`) and negative words (`terrible`, `awful`, `hate`)
- Count positive vs negative words
- **Problem**: Can't handle sarcasm, context, or complex sentences
- Example: *"Not bad"* → Rule-based thinks it's negative (because of "bad"), but it's actually positive!

### Approach 2: ML Classification (Traditional ML)
- Train a model (Naive Bayes, SVM, etc.) on labeled data
- Each review is labeled: positive/negative/neutral
- Model learns patterns from the labeled data
- **Problem**: Needs lots of labeled training data

### Approach 3: LLM-Based (What We Use!) ✅
- Just ask the LLM: *"What is the sentiment of this review?"*
- LLM already understands language, sarcasm, context
- **No training data needed** — works out of the box
- Can also extract WHY (pros, cons, issues) — not just positive/negative

```
Traditional:  "Is this positive or negative?"  → Positive
LLM-Based:    "Analyze this review"             → Positive, Rating: 4.2/5,
                                                    Pros: [fast, good display],
                                                    Cons: [heavy, loud fan]
```

---

## How We Do Sentiment Analysis in This Project

Our approach (LLM + Pydantic structured output):

```
Step 1: Search the web for reviews of a product
        → "ASUS ROG Strix G16 review pros cons 2026"

Step 2: Feed the raw review text to the LLM
        → "Analyze these reviews..."

Step 3: LLM returns structured analysis (via Pydantic):
        {
            "product_name": "ASUS ROG Strix G16",
            "overall_sentiment": "Positive",
            "average_rating": "4.3",
            "pros": ["Powerful GPU", "Good display", "Fast SSD"],
            "cons": ["Heavy", "Loud fans", "Average battery"],
            "common_issues": ["Overheating under heavy load"],
            "expert_opinion": "Great for gaming and AI workloads..."
        }
```

### Why is this better than just reading reviews?
- **Speed**: Analyzes hundreds of reviews in seconds
- **Consistency**: Same criteria applied to every product
- **Structure**: Outputs pros/cons/issues in a format the next agent can consume
- **Aggregation**: Combines opinions from multiple sources into one summary

In [ ]:
# ============================================================
# EXPERIMENT: Try sentiment analysis yourself with the LLM!
# ============================================================

# Let's manually test sentiment analysis on sample reviews

from pydantic import BaseModel, Field
from typing import List

# Define a simple sentiment schema
class SentimentResult(BaseModel):
    text: str = Field(description="The original review text")
    sentiment: str = Field(description="Positive, Negative, or Mixed")
    confidence: float = Field(description="Confidence score 0.0 to 1.0")
    key_phrases: List[str] = Field(description="Key phrases that indicate sentiment")

# Create a simple sentiment chain
sentiment_chain = llm.with_structured_output(SentimentResult)

# Test with sample reviews
sample_reviews = [
    "This laptop is incredible! The RTX 4060 handles AI workloads like a charm. Battery could be better though.",
    "Terrible experience. Screen flickered after 2 weeks. Customer support was unhelpful.",
    "It's decent for the price. Nothing extraordinary but gets the job done for basic ML tasks.",
]

print("=" * 60)
print("SENTIMENT ANALYSIS EXPERIMENT")
print("=" * 60)

for review in sample_reviews:
    result = sentiment_chain.invoke(
        f"Analyze the sentiment of this review:\n\n{review}"
    )
    print(f"\n📝 Review: {result.text[:60]}...")
    print(f"   Sentiment  : {result.sentiment}")
    print(f"   Confidence : {result.confidence}")
    print(f"   Key Phrases: {result.key_phrases}")
    print("-" * 60)

### What just happened?

1. We defined a `SentimentResult` Pydantic schema — telling the LLM to return sentiment + confidence + key phrases
2. We used `llm.with_structured_output(SentimentResult)` — same pattern as our agents
3. We passed each review to the LLM — it analyzed the text and returned structured results

**This is exactly what Agent 2 does**, but at scale — for multiple products, across multiple review sources.

### Experiment Ideas:
- Try adding your own reviews to `sample_reviews`
- Try sarcastic reviews: `"Oh great, another laptop that overheats. Just what I needed."`
- Try reviews in Hinglish: `"Laptop bahut accha hai, but battery kharab hai"`

---
## Types of Sentiment Analysis

| Type | What it does | Example |
|------|-------------|---------|
| **Binary** | Positive or Negative only | "Is this review good or bad?" |
| **Ternary** | Positive, Negative, or Neutral | "Is this good, bad, or meh?" |
| **Fine-grained** | 1-5 star rating | "Rate this review 1-5" |
| **Aspect-based** | Sentiment per feature | "Battery: Negative, Display: Positive" |
| **Emotion detection** | Specific emotions | "Angry, Happy, Frustrated, Excited" |

### In our project, we use a combination:
- **Ternary sentiment** (Positive / Mixed / Negative)
- **Fine-grained rating** (estimated X/5)
- **Aspect-based extraction** (pros = positive aspects, cons = negative aspects)

This gives us a **complete picture** of what users think about each product.

---
Now let's apply this in **Agent 2** below.

---
# PART 6: Agent 2 — Review Analysis Agent

### What does this agent do?
Searches reviews → LLM performs **sentiment analysis** (as we learned above) → extracts pros, cons, issues

### Why reviews matter
Specs don't tell you about overheating, build quality, or real battery life. Reviews do.

### How Agent 2 uses sentiment analysis:
1. Searches web for reviews of each product
2. Feeds all review text to LLM
3. LLM performs **ternary sentiment** (Positive/Mixed/Negative)
4. LLM performs **aspect-based extraction** (pros, cons, issues)
5. LLM summarizes **expert opinions**
6. All output is structured via `ReviewOutput` Pydantic schema

In [ ]:
# BUILD: Review Agent
review_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a review analysis expert. For each product determine:
sentiment (Positive/Mixed/Negative), rating/5, pros, cons, issues, expert opinion.
Base analysis on provided data. If limited, note it.
"""),
    ("human", "Products:\n{product_list}\n\nReview Data:\n{review_data}")
])
review_chain = review_prompt | llm.with_structured_output(ReviewOutput)
print("Review Agent built!")

In [ ]:
# RUN: Review Agent
print("[Agent 2] Analyzing reviews...\n")

all_reviews = ""
for p in discovery_result.products:
    name = f"{p.brand} {p.model}"
    print(f"  Searching reviews: {name}")
    all_reviews += f"\n--- {name} ---\n{search_reviews(name)}\n"

review_result = review_chain.invoke({"product_list": product_list_str, "review_data": all_reviews})

print("\n" + "=" * 70 + "\nREVIEW ANALYSIS\n" + "=" * 70)
for r in review_result.reviews:
    print(f"\n{r.product_name} — {r.overall_sentiment} ({r.average_rating}/5)")
    print(f"  Pros: {', '.join(r.pros)}")
    print(f"  Cons: {', '.join(r.cons)}")
    print(f"  Issues: {', '.join(r.common_issues)}")
    print(f"  Expert: {r.expert_opinion}")

---
# PART 7: Agent 3 — Recommendation Agent

### What does this agent do?
Takes **ALL data** from previous agents → scores each product → ranks → gives verdict

### Scoring (0-10 scale, weighted):
- Performance (CPU/GPU) = 30%
- Value for money = 25%
- User reviews = 20%
- AI Readiness (VRAM, RAM) = 25%

### This agent does NOT search the web!
It only **reasons** over data already collected by the other agents.

In [ ]:
# BUILD: Recommendation Agent
rec_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a product recommendation expert. Score each product (0-10):
Performance(30%), Value(25%), Reviews(20%), AI Readiness(25%).
Rank best to worst. Give top pick, trade-offs, verdict, confidence level.
Be objective. Don't hallucinate specs.
"""),
    ("human", "User: {user_query}\n\nProducts:\n{discovery_data}\n\nSpecs:\n{spec_data}\n\nReviews:\n{review_data}")
])
recommendation_chain = rec_prompt | llm.with_structured_output(RecommendationOutput)
print("Recommendation Agent built!")

In [ ]:
# RUN: Recommendation Agent
print("[Agent 3] Generating recommendation...\n")

# Prepare summary strings from all previous agents
disc_str = "\n".join(f"- {p.brand} {p.model}: {p.specs_hint} ({p.reason})" for p in discovery_result.products)
spec_str = "\n".join(f"- {s.product_name}: CPU={s.cpu}, GPU={s.gpu}, RAM={s.ram}, Price={s.price}" for s in spec_result.specs)
rev_str = "\n".join(f"- {r.product_name}: {r.overall_sentiment}, {r.average_rating}/5, Pros={r.pros[:3]}, Cons={r.cons[:3]}" for r in review_result.reviews)

recommendation_result = recommendation_chain.invoke({
    "user_query": user_query, "discovery_data": disc_str,
    "spec_data": spec_str, "review_data": rev_str
})

# Display
print("=" * 70 + "\nFINAL RECOMMENDATION REPORT\n" + "=" * 70)
print(f"\n🏆 TOP PICK: {recommendation_result.top_pick}")
print(f"   {recommendation_result.top_pick_reason}")
print(f"\n📊 RANKINGS:")
for s in sorted(recommendation_result.scores, key=lambda x: x.rank):
    print(f"  #{s.rank} {s.product_name} — {s.overall_score}/10 (Perf:{s.performance_score} Val:{s.value_score} Rev:{s.review_score} AI:{s.ai_readiness_score})")
print(f"\n⚖️  TRADE-OFFS: {recommendation_result.trade_offs}")
print(f"✅ VERDICT: {recommendation_result.final_verdict}")
print(f"🎯 CONFIDENCE: {recommendation_result.confidence}")

---
# PART 8: Full Pipeline — One Function

### What is orchestration?
Running all 4 agents in sequence — each agent's output feeds into the next.

```python
result = run_product_recommendation("Best phones under 30000 INR")
```

In [ ]:
def run_product_recommendation(query: str) -> RecommendationOutput:
    """Run ALL 4 agents in sequence. Returns final recommendation."""
    print("=" * 70 + f"\nQuery: {query}\n" + "=" * 70)

    # Agent 0
    print("\n[Agent 0] Discovering...")
    ctx = search_products(f"{query} top models India {date.today().year}", 10)
    disc = discovery_chain.invoke({"user_query": query, "search_context": ctx})
    for p in disc.products: print(f"  - {p.brand} {p.model}")

    # Agent 1
    print("\n[Agent 1] Specs...")
    pl = "\n".join(f"- {p.brand} {p.model}" for p in disc.products)
    rd = ""
    for p in disc.products:
        n = f"{p.brand} {p.model}"; print(f"  {n}")
        rd += f"\n--- {n} ---\n{search_specs(n)}\n"
    sp = spec_chain.invoke({"product_list": pl, "research_data": rd})

    # Agent 2
    print("\n[Agent 2] Reviews...")
    rv = ""
    for p in disc.products:
        n = f"{p.brand} {p.model}"; print(f"  {n}")
        rv += f"\n--- {n} ---\n{search_reviews(n)}\n"
    rev = review_chain.invoke({"product_list": pl, "review_data": rv})

    # Agent 3
    print("\n[Agent 3] Recommending...")
    ds = "\n".join(f"- {p.brand} {p.model}: {p.specs_hint}" for p in disc.products)
    ss = "\n".join(f"- {s.product_name}: CPU={s.cpu}, GPU={s.gpu}, RAM={s.ram}, Price={s.price}" for s in sp.specs)
    rs = "\n".join(f"- {r.product_name}: {r.overall_sentiment}, {r.average_rating}/5" for r in rev.reviews)
    rec = recommendation_chain.invoke({"user_query": query, "discovery_data": ds, "spec_data": ss, "review_data": rs})

    print("\n" + "=" * 70 + "\nFINAL REPORT\n" + "=" * 70)
    print(f"🏆 {rec.top_pick} — {rec.top_pick_reason}")
    for s in sorted(rec.scores, key=lambda x: x.rank):
        print(f"  #{s.rank} {s.product_name} — {s.overall_score}/10")
    print(f"⚖️  {rec.trade_offs}")
    print(f"✅ {rec.final_verdict}")
    print(f"🎯 Confidence: {rec.confidence}")
    return rec

print("Pipeline ready!")

In [ ]:
# RUN the full pipeline
result = run_product_recommendation("Best laptops for AI development under 150000 INR")

In [ ]:
# EXPERIMENT: Access results
print("Top:", result.top_pick)
print("Confidence:", result.confidence)
print("Dict:", result.model_dump())

### Experiment: Try different queries!
```python
run_product_recommendation("Best headphones under 5000 INR")
run_product_recommendation("Best monitor for coding under 25000 INR")
run_product_recommendation("Best mobile under 20000 INR with good camera")
```